# Detecção e diagnóstico de anomalias em sinais de vibração (CWRU)

ECM514 — Ciência de Dados. Pipeline ponta a ponta, executável no Colab do zero.

A lógica mora em `src/`; este notebook só orquestra. Decisões e justificativas
estão no relatório (`docs/relatorio/`).

## 0. Ambiente

No Colab, clona o repositório (os `.mat` do CWRU estão versionados nele) e
instala as dependências. Localmente, basta abrir o notebook a partir da raiz
do repositório ou de `notebooks/`.

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/Nyfeu/Vibration_Analysis.git"
if "google.colab" in sys.modules:
    if not os.path.isdir("Vibration_Analysis"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("Vibration_Analysis")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print(os.getcwd())

In [ ]:
from src.data import (
    SEED, FS_HZ, fixar_seeds, ler_manifesto, montar_janelas,
    split_por_carga, contagem_janelas, tamanho_janela,
)
rng = fixar_seeds(SEED)

## 1. Dados e protocolo

Leitura do canal DE pela variável do próprio arquivo, decimação 48 → 12 kHz dos
normais, banda comum de 0 a 4,8 kHz em todas as gravações, janelas de 4096
amostras (3 períodos da gaiola na menor rotação) com 50% de sobreposição.

In [ ]:
manifesto = ler_manifesto()
print("menor rotação:", manifesto["rpm"].min(), "rpm -> janela de",
      tamanho_janela(manifesto["rpm"].min()), "amostras")

X, meta = montar_janelas("principal")
treino, teste = split_por_carga(meta)  # falha se uma gravação cair nos dois lados
print(X.shape, "| treino:", treino.sum(), "janelas | teste:", teste.sum(), "janelas")
contagem_janelas(meta)

In [ ]:
# Tabelas e figuras de dados usadas no relatório (results/metrics, results/figures).
subprocess.run([sys.executable, "-m", "scripts.gerar_artefatos_dados"], check=True)

In [ ]:
# Testes automatizados: frequências do rolamento, features, decimação, split.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)

## 2. Features

Domínio do tempo (RMS, curtose de Pearson, fator de crista, assimetria, pico)
e escores do espectro de envelope (BPFO, BPFI, BSF, em dB acima do fundo local),
uma linha por janela.

In [ ]:
import pandas as pd
from src.features import features_tempo, features_envelope

F = pd.concat(
    [features_tempo(X), features_envelope(X, meta["rpm"].to_numpy(), FS_HZ)], axis=1
)
F.groupby(meta["classe"].to_numpy()).median().round(2)

## 3. Validação física

O pico do envelope cai na frequência prevista pela teoria? Critério e
comparação com Smith & Randall (2015, Tab. B2) em `src/evaluation.py`.

In [ ]:
from src.evaluation import validacao_fisica, resumo_validacao

subprocess.run([sys.executable, "-m", "scripts.gerar_artefatos_envelope"], check=True)
resumo_validacao(validacao_fisica())

## 4. Detecção one-class

TODO (issues #15–#17).

## 5. Classificação

TODO (issues #19–#20).